# Generate an Antibody Binder with peleke-1

(Note: This notebook is designed for Google Colab. If you are running it locally, make sure to install the required dependencies and change the paths accordingly. Also, you may need an A100 GPU runtime.)

In [4]:
pip install -qq torchao -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.9 MB/s eta 0:00:00


## Download the model and tokenizer

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig
import torch, re

In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
model_name = 'silicobio/peleke-llama-3.1-8b-instruct'
config = PeftConfig.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, torch_dtype=torch.bfloat16, trust_remote_code=True).cuda()
model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(model, model_name).cuda()



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

## Create the antibody

peleke-1 model use annotated antigen sequences to produce targeted antibodies. In the string of amino acids in the antigen, we surround each epitope residue with square brackets (`[X]`). Then, we use a function to convert these to special `<epi>` and `</epi>` tokens.


In [6]:
def format_prompt(antigen_sequence):
    epitope_seq = re.sub(r'\[([A-Z])\]', r'<epi>\1</epi>', antigen_sequence)
    formatted_str = f"Antigen: {epitope_seq}<|im_end|>\nAntibody:"
    return formatted_str

In [7]:
## Example ## PD-1 antigen sequence with epitope residues marked
antigen_sequence = "NPPTFSPALLVVTEGDNATFTCSFS[S][F][V]L[N]WYRMQ[T][D][K]LAAF[P]E[D][R][S][Q][P][G]QDSRFRVTQLPNGRDFHMSVVRARRNDSGTYLCGA[I]S[L]AQIKESLRAELRV"

prompt = format_prompt(antigen_sequence)
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

In [8]:
full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
antibody_sequence = full_text.split('<|im_end|>')[1].replace('Antibody: ', '')
print(f"Antigen: {antigen_sequence}\nAntibody: {antibody_sequence}\n")

Antigen: NPPTFSPALLVVTEGDNATFTCSFS[S][F][V]L[N]WYRMQ[T][D][K]LAAF[P]E[D][R][S][Q][P][G]QDSRFRVTQLPNGRDFHMSVVRARRNDSGTYLCGA[I]S[L]AQIKESLRAELRV
Antibody: 
EVQLVESGGGLVQPGGSLRLSCAASGFNVSYSSIHWVRQAPGKGLEWVAYIYSYSGSTYYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARGGSFAYWGQGTLVTVSS|DIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYKYVPVTFGQGTKVEIK



## Evaluate the Antibody-Antigen Complex

Using your favorite folding tool, fold the antibody and antigen sequences together. Then, evaluate if the model produced an antibody that binds to the desired site.

In [10]:
pip install py3Dmol

In [11]:
import py3Dmol

In [17]:
cif_path = '/content/fold_jcsu_peleke_ll8_pd_1_001_scfv_model_0.cif'

with open(cif_path, "r") as f:
        cif_data = f.read()

## Create viewer
view = py3Dmol.view(
    width=1200,
    height=800
)

view.addModel(cif_data, "cif")

## Show binder in yellow
view.setStyle(
    {'chain':'A'},{'cartoon': {'color': 'yellow'}}
)

## Show target in cyan
view.setStyle(
    {'chain':'B'},{'cartoon': {'color': 'cyan'}}
)

view.zoomTo()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Try Again!

Now that you've seen how the *peleke-1* model works, find another target antigen from Protein Data Bank and try to design an antibody against it. You'll need to select the desired epitope residues and annotate them with `[X]`.